## **LSTM : Hotel Review Sentiment Classification**
**Individual contribution: Ummu Salma M. H. - IT23190870**

Task: classify a hotel review as **positive (1)** or **negative (0)**.

In [48]:
# Imports and dataset location 
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve)

layers = keras.layers

# Locate dataset and split files locally
DATA_NAME = "deceptive-opinion.csv"
SPLIT_NAME = "split_assignment.csv"
here = pathlib.Path.cwd().resolve()
search_roots = [here, here.parent, here.parent.parent]

data_candidates = [r / "dataset" / DATA_NAME for r in search_roots] + [r / DATA_NAME for r in search_roots]
DATA_PATH = next((p for p in data_candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Cannot find {DATA_NAME}. Ensure dataset/{DATA_NAME} exists.")

split_candidates = [r / "shared" / SPLIT_NAME for r in search_roots] + [r / SPLIT_NAME for r in search_roots]
SPLIT_PATH = next((p for p in split_candidates if p.exists()), None)
if SPLIT_PATH is None:
    raise FileNotFoundError(f"Cannot find {SPLIT_NAME}. Ensure shared/{SPLIT_NAME} exists.")

# Root folder for LSTM
LSTM_ROOT = here
RESULTS = LSTM_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("Dataset:", DATA_PATH)
print("Split file:", SPLIT_PATH)
print("Results folder:", RESULTS)


Dataset: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\dataset\deceptive-opinion.csv
Split file: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\shared\split_assignment.csv
Results folder: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\lstm\results


In [49]:
# Group settings
SEED = 42

# Labels
LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

# Text -> integer sequences
VOCAB_SIZE = 5000 # total no of ids
MAX_LEN = 300 # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre" # padding at begining
TRUNCATING = "post" # long reviews keep their first MAX_LEN tokens

# Training
EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5 # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5 # probability cut-off used for every model's test metrics

# Split file loaded directly from shared/split_assignment.csv
split_df = pd.read_csv(SPLIT_PATH)
VAL_IDS = set(split_df.loc[split_df["split"] == "val", "row_id"])
TEST_IDS = set(split_df.loc[split_df["split"] == "test", "row_id"])

keras.utils.set_random_seed(SEED) # seeds Python, NumPy and TensorFlow
print("TensorFlow", tf.__version__, ", Keras", keras.__version__, ", seed", SEED)
print("Devices:", [d.device_type for d in tf.config.list_physical_devices()])
print(f"Loaded fixed split: {len(VAL_IDS)} validation rows, {len(TEST_IDS)} test rows.")


TensorFlow 2.21.0 , Keras 3.15.1 , seed 42
Devices: ['CPU']
Loaded fixed split: 160 validation rows, 160 test rows.


## LSTM Model & Architecture Comparison (Unidirectional vs. Bidirectional)

### Objective

Compare a **unidirectional LSTM** with a **bidirectional LSTM** for hotel review sentiment classification.

Both models use the same preprocessing, data split, hyperparameters, and training procedure. The architectural difference is how the sequence is processed.

### Architecture

Input (300 token IDs)
→ Embedding(5000, 64)
→ LSTM(64)
→ Dropout(0.5)
→ Dense(32, ReLU)
→ Dropout(0.3)
→ Dense(1, Sigmoid)

### Unidirectional LSTM

Processes the sequence in one direction:

Input → → → → →

### Bidirectional LSTM

Processes the sequence in both forward and backward directions:

Forward:  → → → → →
Backward: ← ← ← ← ←

The forward and backward representations are combined before the classification layers.

In [62]:
def build_unidirectional_lstm(units=64, embed_dim=EMBED_DIM, dropout1=0.5, dense_units=32, dropout2=0.3):
    """Unidirectional LSTM: processes sequence strictly forward (past to future)."""
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, embed_dim),                
        layers.LSTM(units, return_sequences=False),             
        layers.Dropout(dropout1),                               
        layers.Dense(dense_units, activation="relu"),           
        layers.Dropout(dropout2),                               
        layers.Dense(1, activation="sigmoid"), # Binary output probability P(positive)
    ], name="unidirectional_lstm_sentiment")

def build_bidirectional_lstm(units=64, embed_dim=EMBED_DIM, dropout1=0.5, dense_units=32, dropout2=0.3):
    """Bidirectional LSTM: processes sequence in both forward and backward directions."""
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, embed_dim),                
        layers.Bidirectional(layers.LSTM(units, return_sequences=False)), 
        layers.Dropout(dropout1),                               
        layers.Dense(dense_units, activation="relu"),           
        layers.Dropout(dropout2),                              
        layers.Dense(1, activation="sigmoid"), # Binary output probability P(positive)
    ], name="bidirectional_lstm_sentiment")

# Default build function 
def build_lstm():
    return build_unidirectional_lstm()

def n_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

# Instantiate and inspect both architectures
uni_model = build_unidirectional_lstm()
bi_model = build_bidirectional_lstm()

print("=" * 75)
print("1. UNIDIRECTIONAL LSTM SUMMARY")
print("=" * 75)
uni_model.summary()

print("\n" + "=" * 75)
print("2. BIDIRECTIONAL LSTM SUMMARY")
print("=" * 75)
bi_model.summary()

uni_bd = {l.name: n_params(l.trainable_weights) for l in uni_model.layers if l.trainable_weights}
bi_bd = {l.name: n_params(l.trainable_weights) for l in bi_model.layers if l.trainable_weights}
print(f"\nUnidirectional Trainable params: {uni_bd} , Total: {uni_model.count_params():,}")
print(f"Bidirectional Trainable params: {bi_bd} , Total: {bi_model.count_params():,}")
print(f"Embedding share of parameters: {100 * next((v for k, v in uni_bd.items() if 'embedding' in k), 320000) / uni_model.count_params():.1f}%")


1. UNIDIRECTIONAL LSTM SUMMARY


Model: "unidirectional_lstm_sentiment"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 300, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 355,137 (1.35 MB)

 Trainable params: 355,137 (1.35 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 300, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,209 (1.49 MB)

 Trainable params: 390,209 (1.49 MB)

 Non-trainable params: 0 (0.00 B)


Unidirectional Trainable params: {'embedding_2': 320000, 'lstm_2': 33024, 'dense_4': 2080, 'dense_5': 33} , Total: 355,137
Bidirectional Trainable params: {'embedding_3': 320000, 'bidirectional_1': 66048, 'dense_6': 4128, 'dense_7': 33} , Total: 390,209
Embedding share of parameters: 90.1%
